# Network Intrusion Detection: v8 (multi-family stage 2)

Two-stage model: stage 1 detects Availability from stable packet features; stage 2 uses LightGBM + XGBoost + CatBoost, compared with and without invariant ratio features over 5 cluster-grouped splits. Writes `submission_v8_ensemble.csv` and `submission_v8_calibrated.csv`.


In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix, precision_score, recall_score
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# Try importing CatBoost and XGBoost (pre-installed in standard Kaggle Python images)
try:
    from catboost import CatBoostClassifier
    HAS_CB = True
except ImportError:
    HAS_CB = False

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

SEED = 42
TARGET = "Target"
ID = "id"
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
T0 = time.time()

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"./{name}") + glob.glob(f"./data/{name}") + glob.glob(f"../data/{name}") + glob.glob(f"../../data/{name}")
    if not hits:
        raise FileNotFoundError(f"{name} not found.")
    return hits[0]

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
sample = pd.read_csv(find("sample_submission.csv"))
print(f"train: {train.shape} | test: {test.shape} | HAS_CB: {HAS_CB} | HAS_XGB: {HAS_XGB}")

## 1. Target & Base Feature Preparation

In [ ]:
classes = sorted(train[TARGET].unique())            # Availability, Benign, Reconnaissance, Spoofing
cls2i = {c: i for i, c in enumerate(classes)}
K = len(classes)
y = train[TARGET].map(cls2i).values
train_mix = train[TARGET].value_counts(normalize=True).reindex(classes)

AV = cls2i["Availability"]
is_av = (y == AV)
rest_classes = [c for c in classes if c != "Availability"]
K3 = len(rest_classes)
y3 = np.full(len(y), -1)
for j, c in enumerate(rest_classes):
    y3[y == cls2i[c]] = j
nonav_idx = np.where(~is_av)[0]

def prepare(train_df, test_df, drop=()):
    feats = [c for c in train_df.columns if c not in (ID, TARGET) and c not in drop]
    full = pd.concat([train_df[feats], test_df[feats]], ignore_index=True)
    full = full.replace([np.inf, -np.inf], np.nan)
    for c in full.select_dtypes(exclude="number").columns:
        full[c] = full[c].astype("category")
    return full.iloc[:len(train_df)].reset_index(drop=True), full.iloc[len(train_df):].reset_index(drop=True)

X_tr, X_te = prepare(train, test)
PORTS = [c for c in ["id.orig_p", "id.resp_p"] if c in X_tr.columns]
num_cols = X_tr.select_dtypes("number").columns.tolist()
beh_cols = [c for c in num_cols if c not in PORTS]

## 2. Shift-Aware Cluster Folds (Stage 1 & Stage 2)

In [ ]:
Z = X_tr[beh_cols].copy()
Z = Z.fillna(Z.median())
Z = np.sign(Z) * np.log1p(Z.abs())
Z = StandardScaler().fit_transform(Z)
Z = PCA(n_components=min(20, Z.shape[1]), random_state=SEED).fit_transform(Z)

groups = np.zeros(len(y), dtype=int)
offset = 0
for ci, cls in enumerate(classes):
    idx = np.where(y == ci)[0]
    k = int(np.clip(len(idx) // 1500, 6, 20))
    km = MiniBatchKMeans(n_clusters=k, random_state=SEED, n_init=3, batch_size=4096).fit(Z[idx])
    groups[idx] = km.labels_ + offset
    offset += k

folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr, y, groups))

## 3. Stage 1: Availability Detection (Locked from v4/v7)

In [ ]:
A = list(X_tr.columns)
B = [c for c in A if c not in PORTS]
TIME_RATE = [c for c in B if ("per_sec" in c or "per_second" in c or "_iat." in c or c.startswith(("active.", "idle."))
                              or c == "flow_duration" or "bulk_rate" in c)]
RESPONSE = [c for c in B if c.startswith("bwd_") or c in ("down_up_ratio", "flow_RST_flag_count", "flow_ACK_flag_count")]
D = [c for c in B if c not in TIME_RATE]
E = [c for c in D if c not in RESPONSE]

lgb_params = dict(n_estimators=250, learning_rate=0.05, num_leaves=15, min_child_samples=100,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=5.0,
                  random_state=SEED, n_jobs=-1, verbose=-1)

# Fit Stage 1 on Set E (no ports, no timing/rate/response - lowest adversarial shift)
s1_feats = E
m1 = lgb.LGBMClassifier(**lgb_params).fit(X_tr[s1_feats], is_av.astype(int))
p_av_test = m1.predict_proba(X_te[s1_feats])[:, 1]
print(f"Stage 1 (Set E) flagged {(p_av_test >= 0.5).sum()} test rows as Availability ({(p_av_test >= 0.5).mean():.1%})")

## 4. Stage 2 Feature Engineering: Rate-Invariant Flow Ratios

In [ ]:
def add_stage2_features(df):
    d = df.copy()
    eps = 1e-5
    # Packet & directional symmetry ratios (crucial for Reconnaissance vs Spoofing)
    if "fwd_pkts_tot" in d and "bwd_pkts_tot" in d:
        tot_pkts = d["fwd_pkts_tot"] + d["bwd_pkts_tot"] + eps
        d["fe_fwd_pkt_share"] = d["fwd_pkts_tot"] / tot_pkts
        d["fe_pkt_diff_norm"] = (d["fwd_pkts_tot"] - d["bwd_pkts_tot"]) / tot_pkts
    
    # Flag density per packet (invariant to scan speed / duration shift)
    if "fwd_pkts_tot" in d and "bwd_pkts_tot" in d:
        tot_pkts = d["fwd_pkts_tot"] + d["bwd_pkts_tot"] + eps
        for flag in ["flow_SYN_flag_count", "flow_RST_flag_count", "flow_ACK_flag_count", "flow_FIN_flag_count"]:
            if flag in d:
                d[f"fe_{flag}_rate"] = d[flag] / tot_pkts

    # Header vs Payload proportions
    if "fwd_header_size_tot" in d and "bwd_header_size_tot" in d:
        d["fe_header_ratio"] = (d["fwd_header_size_tot"] + 1.0) / (d["bwd_header_size_tot"] + 1.0)
    
    if "fwd_pkts_payload.tot" in d and "bwd_pkts_payload.tot" in d:
        tot_pay = d["fwd_pkts_payload.tot"] + d["bwd_pkts_payload.tot"] + eps
        d["fe_fwd_payload_share"] = d["fwd_pkts_payload.tot"] / tot_pay
        if "fwd_pkts_tot" in d and "bwd_pkts_tot" in d:
            d["fe_bytes_per_pkt"] = tot_pay / (d["fwd_pkts_tot"] + d["bwd_pkts_tot"] + eps)

    # Burstiness / IAT CV (std / mean is scale-invariant to clock/rate stretching!)
    if "flow_iat.std" in d and "flow_iat.avg" in d:
        d["fe_flow_iat_cv"] = d["flow_iat.std"] / (d["flow_iat.avg"].abs() + eps)
    if "fwd_iat.std" in d and "fwd_iat.avg" in d:
        d["fe_fwd_iat_cv"] = d["fwd_iat.std"] / (d["fwd_iat.avg"].abs() + eps)
        
    # Port structure hints (ephemeral vs well-known port tiers)
    if "id.resp_p" in d:
        d["fe_resp_well_known"] = (d["id.resp_p"] < 1024).astype(float)
        d["fe_resp_ephemeral"] = (d["id.resp_p"] >= 49152).astype(float)
    return d

Xs_raw = X_tr.iloc[nonav_idx].reset_index(drop=True)
ys = y3[nonav_idx]
Xte_raw = X_te.reset_index(drop=True)

Xs_fe = add_stage2_features(Xs_raw)
Xte_fe = add_stage2_features(Xte_raw)
print(f"Stage 2 features: Base={Xs_raw.shape[1]} | With Invariant Ratios={Xs_fe.shape[1]}")

## 5. Multi-Family Stage 2 Model Suite (LGBM + XGB + CatBoost)

In [ ]:
def sample_w(y_tr, power, k):
    cnt = np.bincount(y_tr, minlength=k).astype(float)
    cnt[cnt == 0] = 1.0
    w = (len(y_tr) / (k * cnt)) ** power
    return w[y_tr]

# The 3 proven v7 LightGBM configs
lgb_cfgs = [
    (dict(n_estimators=600, learning_rate=0.05, num_leaves=15, min_child_samples=80, subsample=0.8, subsample_freq=1,
          colsample_bytree=0.9, reg_lambda=20.0, max_depth=-1, random_state=SEED, n_jobs=-1, verbose=-1), 0.0),
    (dict(n_estimators=150, learning_rate=0.08, num_leaves=4, min_child_samples=5, subsample=0.6, subsample_freq=1,
          colsample_bytree=0.9, reg_lambda=5.0, max_depth=6, random_state=SEED, n_jobs=-1, verbose=-1), 0.0),
    (dict(n_estimators=400, learning_rate=0.03, num_leaves=4, min_child_samples=5, subsample=0.8, subsample_freq=1,
          colsample_bytree=0.9, reg_lambda=1.0, max_depth=4, random_state=SEED, n_jobs=-1, verbose=-1), 0.5),
]

def fit_predict_suite(X_tr_df, y_tr_arr, X_va_df):
    preds = []
    
    # 1. LightGBM models (3x)
    for params, power in lgb_cfgs:
        m = lgb.LGBMClassifier(**params).fit(X_tr_df, y_tr_arr, sample_weight=sample_w(y_tr_arr, power, K3))
        p = np.zeros((len(X_va_df), K3))
        p[:, m.classes_] = m.predict_proba(X_va_df)
        preds.append(p)
        
    # Prepare numeric-encoded copy for XGBoost / CatBoost
    cat_cols = X_tr_df.select_dtypes(exclude="number").columns.tolist()
    X_tr_num, X_va_num = X_tr_df.copy(), X_va_df.copy()
    for c in cat_cols:
        X_tr_num[c] = X_tr_num[c].cat.codes
        X_va_num[c] = X_va_num[c].cat.codes
        
    # 2. XGBoost models (2x)
    if HAS_XGB:
        xgb_cfgs = [
            (dict(n_estimators=350, learning_rate=0.04, max_depth=4, min_child_weight=10,
                  subsample=0.8, colsample_bytree=0.8, reg_lambda=10.0, random_state=SEED, n_jobs=-1, eval_metric="mlogloss"), 0.25),
            (dict(n_estimators=500, learning_rate=0.03, max_depth=3, min_child_weight=5,
                  subsample=0.7, colsample_bytree=0.9, reg_lambda=5.0, random_state=SEED+1, n_jobs=-1, eval_metric="mlogloss"), 0.5),
        ]
        for params, power in xgb_cfgs:
            m = xgb.XGBClassifier(**params).fit(X_tr_num, y_tr_arr, sample_weight=sample_w(y_tr_arr, power, K3))
            preds.append(m.predict_proba(X_va_num))
            
    # 3. CatBoost models (2x oblivious trees)
    if HAS_CB:
        cb_cfgs = [
            (dict(iterations=600, learning_rate=0.05, depth=5, l2_leaf_reg=10.0, random_seed=SEED, verbose=0), 0.25),
            (dict(iterations=450, learning_rate=0.06, depth=4, l2_leaf_reg=5.0, random_seed=SEED+1, verbose=0), 0.5),
        ]
        for params, power in cb_cfgs:
            m = CatBoostClassifier(**params).fit(X_tr_num, y_tr_arr, sample_weight=sample_w(y_tr_arr, power, K3))
            preds.append(m.predict_proba(X_va_num))
            
    return preds # [lgb1, lgb2, lgb3, xgb1, xgb2, cb1, cb2]

## 6. Rigorous Validation Across 5 Fresh Cluster-Grouped Splits

In [ ]:
Zs = Z[nonav_idx]

def make_folds(seed, k_per_class=15):
    rng_ = np.random.default_rng(seed)
    fold_of = np.zeros(len(ys), dtype=int)
    for c in range(K3):
        idx = np.where(ys == c)[0]
        k = int(min(k_per_class, max(5, len(idx) // 40)))
        km = MiniBatchKMeans(n_clusters=k, random_state=seed, n_init=3, batch_size=2048).fit(Zs[idx])
        order = rng_.permutation(k)
        cl_fold = {cl: pos % 5 for pos, cl in enumerate(order)}
        fold_of[idx] = [cl_fold[l] for l in km.labels_]
    return [(np.where(fold_of != f)[0], np.where(fold_of == f)[0]) for f in range(5)]

def optimize_thresholds(oof_probs, y_true):
    w = np.ones(K3)
    best = f1_score(y_true, oof_probs.argmax(1), average="macro")
    grid = np.linspace(0.6, 1.5, 37)
    for _ in range(3):
        for k in range(K3):
            for g in grid:
                w2 = w.copy()
                w2[k] = g
                s = f1_score(y_true, (oof_probs * w2).argmax(1), average="macro")
                if s > best + 1e-5:
                    best, w = s, w2
    return w

seeds = list(range(10, 15))
oof_v7_all, oof_v8_raw_all, oof_v8_fe_all = [], [], []

for sd in seeds:
    n_models = 3 + (2 if HAS_XGB else 0) + (2 if HAS_CB else 0)
    oof_raw = np.zeros((n_models, len(ys), K3))
    oof_fe  = np.zeros((n_models, len(ys), K3))
    
    for tr, va in make_folds(sd):
        pr_raw = fit_predict_suite(Xs_raw.iloc[tr], ys[tr], Xs_raw.iloc[va])
        pr_fe  = fit_predict_suite(Xs_fe.iloc[tr],  ys[tr], Xs_fe.iloc[va])
        for m_i in range(n_models):
            oof_raw[m_i, va] = pr_raw[m_i]
            oof_fe[m_i, va]  = pr_fe[m_i]
            
    oof_v7_all.append(oof_raw[:3].mean(axis=0))        # v7: 3 LGBM on raw features
    oof_v8_raw_all.append(oof_raw.mean(axis=0))        # v8_raw: Multi-family on raw features
    oof_v8_fe_all.append(oof_fe.mean(axis=0))          # v8_fe: Multi-family on engineered features

sc_v7     = [f1_score(ys, p.argmax(1), average="macro") for p in oof_v7_all]
sc_v8_raw = [f1_score(ys, p.argmax(1), average="macro") for p in oof_v8_raw_all]
sc_v8_fe  = [f1_score(ys, p.argmax(1), average="macro") for p in oof_v8_fe_all]

print("\n--- 5-Split Cluster-Grouped CV Comparison (Seeds 10-14) ---")
print(f"v7 (3x LGBM, Set A)             : {np.mean(sc_v7):.4f} +/- {np.std(sc_v7):.4f} | {np.round(sc_v7, 4)}")
print(f"v8_raw (LGB+XGB+CB, Set A)      : {np.mean(sc_v8_raw):.4f} +/- {np.std(sc_v8_raw):.4f} | {np.round(sc_v8_raw, 4)}")
print(f"v8_fe  (LGB+XGB+CB, Set A + FE) : {np.mean(sc_v8_fe):.4f} +/- {np.std(sc_v8_fe):.4f} | {np.round(sc_v8_fe, 4)}")

# Pick whichever feature set (raw vs FE) has higher multi-split CV
use_fe = np.mean(sc_v8_fe) > np.mean(sc_v8_raw)
best_oof_list = oof_v8_fe_all if use_fe else oof_v8_raw_all
X_train_final = Xs_fe if use_fe else Xs_raw
X_test_final  = Xte_fe if use_fe else Xte_raw
print(f"\nSelected feature matrix for v8: {'Set A + Invariant FE' if use_fe else 'Set A (Raw)'}")

# Cross-validated threshold tuning (fit multipliers on 4 seeds, evaluate on held-out seed)
sc_thresh = []
weights_per_seed = []
for i in range(len(seeds)):
    train_oofs = np.vstack([best_oof_list[j] for j in range(len(seeds)) if j != i])
    train_ys   = np.tile(ys, len(seeds) - 1)
    w_opt = optimize_thresholds(train_oofs, train_ys)
    w_shrunk = w_opt ** 0.5  # Half-strength shrinkage for shift safety
    weights_per_seed.append(w_shrunk)
    sc_thresh.append(f1_score(ys, (best_oof_list[i] * w_shrunk).argmax(1), average="macro"))

print(f"v8 + Half-Shrunk Thresholds     : {np.mean(sc_thresh):.4f} +/- {np.std(sc_thresh):.4f} | {np.round(sc_thresh, 4)}")
final_w = np.mean(weights_per_seed, axis=0)
print("Final Stage-2 Probability Multipliers:", {c: round(float(v), 3) for c, v in zip(rest_classes, final_w)})

## 7. Final Fit & Write Submissions

In [ ]:
rest_idx = np.array([cls2i[c] for c in rest_classes])
test_preds_suite = fit_predict_suite(X_train_final, ys, X_test_final)
P_v8 = np.mean(test_preds_suite, axis=0)

# File 1: Multi-family ensemble without threshold multiplier
lab_v8_ens = np.array(classes)[np.where(p_av_test >= 0.5, AV, rest_idx[P_v8.argmax(1)])]
sub_ens = pd.DataFrame({ID: test[ID].values, TARGET: lab_v8_ens})
path_ens = os.path.join(OUT, "submission_v8_ensemble.csv")
sub_ens.to_csv(path_ens, index=False)

# File 2: Multi-family ensemble + OOF-validated probability multipliers
P_v8_cal = P_v8 * final_w
lab_v8_cal = np.array(classes)[np.where(p_av_test >= 0.5, AV, rest_idx[P_v8_cal.argmax(1)])]
sub_cal = pd.DataFrame({ID: test[ID].values, TARGET: lab_v8_cal})
path_cal = os.path.join(OUT, "submission_v8_calibrated.csv")
sub_cal.to_csv(path_cal, index=False)

print(f"\nSaved {path_ens} and {path_cal} [{time.time()-T0:.0f}s]")
print("Class distribution in submission_v8_calibrated.csv:")
print(sub_cal[TARGET].value_counts(normalize=True).reindex(classes).round(3))
print(f"Diff between v8_ensemble and v8_calibrated: {(lab_v8_ens != lab_v8_cal).sum()} rows")